schema: 
CREATE TABLE Students (id int, name TEXT);
CREATE TABLE Takes (sid Int, cid Int);

query1: SELECT Students.name
FROM Students
LEFT JOIN Takes ON Takes.sid = Students.id
WHERE Students.id >= 3

query 2: SELECT Students.name
FROM Students, Takes
WHERE Takes.sid = Students.id OR Students.id >= 3

In [1]:
from z3 import *

# declare "input" variables for each attribute in all tables
Students_present = Bool("Students_present")
Students_id = Int("Students_id")
Students_id_is_null = Bool("Students_id_is_null")
Students_name = String("Students_name")
Students_name_is_null = Bool("Students_name_is_null")


Takes_present = Bool("Takes_present")
Takes_sid = Int("Takes_sid")
Takes_sid_is_null = Bool("Takes_sid_is_null")
Takes_cid = Int("Takes_cid") 
Takes_cid_is_null = Bool("Takes_cid_is_null") 

Encode q1:
SELECT Students.name
FROM Students
left JOIN Takes ON Takes.sid = Students.id
OR Students.id >= 3

In [2]:
q1_constraints = []

# step1: encode left join 
q1_join_cond = And(Not(Takes_sid_is_null), Not(Students_id_is_null), Takes_sid == Students_id)

q1_match = Bool("q1_match")         # matched row (Students,Takes)
q1_right_null = Bool("q1_right_null") # null-extended (Students,NULL)
q1_left_null = Bool("q1_left_null") # null-extended (NULL, Students)
q1_constraints += [
    q1_match == q1_join_cond,
    q1_right_null == And(Students_present, Not(q1_join_cond)),
    q1_left_null == False
]


# step2: declaring variables for variables after join (intermediate representation)
# J1 means after join for query 1
J1_Students_id = Int("J1_Students_id")
J1_Students_id_is_null = Bool("J1_Students_id_is_null")
J1_Students_name = String("J1_Students_name")
J1_Students_name_is_null = Bool("J1_Students_name_is_null")

J1_Takes_sid = Int("J1_Takes_sid")
J1_Takes_sid_is_null = Bool("J1_Takes_sid_is_null")
J1_Takes_cid = Int("J1_Takes_cid") 
J1_Takes_cid_is_null = Bool("J1_Takes_cid_is_null") 


# step3: build a connection between the previous variables with the new ones 
left_table, right_table = "Students", "Takes"
join_type = "LEFT"

# for the left table -- Students
use_base = Or(q1_match, q1_right_null)
# for matched row (Students,Takes) and null-extended (Students,NULL)
# we use the original value for those variables, and their nullablility doesn't change 
# while for the others (null-extended (NULL,Students), which doesn't exist in this query ), 
# we set their value to NULL
q1_constraints.append(J1_Students_id  == If(use_base, Students_id, IntVal(0)))
q1_constraints.append(J1_Students_id_is_null == If(use_base, Students_id_is_null, True))
# repeat this process for all attribute in left table 
q1_constraints.append(J1_Students_name  == If(use_base, Students_name, StringVal("")))
q1_constraints.append(J1_Students_name_is_null == If(use_base, Students_name_is_null, True))


# for the right table -- Takes
use_base = q1_match
# for matched rows (Students,Takes)
# we use the original value for those variables, and their nullablility doesn't change 
# while for the unmatched ones, we set their value to NULL
q1_constraints.append(J1_Takes_sid  == If(use_base, Takes_sid, IntVal(0)))
q1_constraints.append(J1_Takes_sid_is_null == If(use_base, Takes_sid_is_null, True))
# repeat this process for all attribute in left table 
q1_constraints.append(J1_Takes_cid == If(use_base, Takes_cid, IntVal(0)))
q1_constraints.append(J1_Takes_cid_is_null == If(use_base, Takes_cid_is_null, True))

Now for encoding where, instead of using variables from the original table, we will use these results/variables from JOIN.

In [3]:
# Encode where
q1_where_cond = And(Not(J1_Students_id_is_null), J1_Students_id >= IntVal(3))

q1_after_match = Bool("q1_after_match")
q1_after_right_null = Bool("q1_after_right_null")
q1_after_left_null = Bool("q1_after_left_null")

q1_constraints += [
    q1_after_match == And(q1_match, q1_where_cond),
    q1_after_right_null == And(q1_right_null, q1_where_cond),
    q1_after_left_null == And(q1_left_null, q1_where_cond)  
]

Encode query2:

SELECT Students.name
FROM Students, Takes
WHERE Takes.sid = Students.id OR Students.id >= 3

In [4]:
q2_constraints = []

# step1: encode Cartisian product
q2_match = Bool("q2_match")      
q2_right_null = Bool("q2_right_null") 
q2_left_null = Bool("q2_left_null") 
q2_constraints += [
    q2_match == And(Students_present, Takes_present),
    q2_right_null == False,
    q2_left_null == False
]


# step2: declaring variables for variables after join (intermediate representation)
# J1 means after join for query 2
J2_Students_id = Int("J2_Students_id")
J2_Students_id_is_null = Bool("J2_Students_id_is_null")
J2_Students_name = String("J2_Students_name")
J2_Students_name_is_null = Bool("J2_Students_name_is_null")

J2_Takes_sid = Int("J2_Takes_sid")
J2_Takes_sid_is_null = Bool("J2_Takes_sid_is_null")
J2_Takes_cid = Int("J2_Takes_cid") 
J2_Takes_cid_is_null = Bool("J2_Takes_cid_is_null") 


# step3: build a connection between the previous variables with the new ones 
left_table, right_table = "Students", "Takes"
join_type = "CP"

# for the both table, keep variables same if match
use_base = q2_match
# left table
q2_constraints.append(J1_Students_id  == If(use_base, Students_id, IntVal(0)))
q2_constraints.append(J1_Students_id_is_null == If(use_base, Students_id_is_null, True))
q2_constraints.append(J1_Students_name  == If(use_base, Students_name, StringVal("")))
q2_constraints.append(J1_Students_name_is_null == If(use_base, Students_name_is_null, True))
# right table
q2_constraints.append(J2_Takes_sid  == If(use_base, Takes_sid, IntVal(0)))
q2_constraints.append(J2_Takes_sid_is_null == If(use_base, Takes_sid_is_null, True))
q2_constraints.append(J2_Takes_cid == If(use_base, Takes_cid, IntVal(0)))
q2_constraints.append(J2_Takes_cid_is_null == If(use_base, Takes_cid_is_null, True))

In [5]:
# encode where: Takes.sid = Students.id OR Students.id >= 3
q2_where_cond = Or(And(And(Not(J2_Takes_sid_is_null), Not(J2_Students_id_is_null)), J2_Takes_sid == J2_Students_id)
                   , And(Not(J1_Students_id_is_null), J1_Students_id >= IntVal(3)))

q2_after_match = Bool("q2_after_match")
q2_after_right_null = Bool("q2_after_right_null")
q2_after_left_null = Bool("q2_after_left_null")

q2_constraints += [
    q2_after_match == And(q2_match, q2_where_cond),
    q2_after_right_null == And(q2_right_null, q2_where_cond),
    q2_after_left_null == And(q2_left_null, q2_where_cond)  
]

In [6]:
# print(q1_constraints)
# print(q2_constraints)

In [7]:
s = Solver()
s.add(q1_constraints + q2_constraints)

diff = Or(
    q1_after_match != q2_after_match,
    q1_after_right_null != q2_after_right_null, 
    q1_after_left_null != q2_after_left_null,
)

s.add(diff)

print("SAT? ->", s.check())
if s.check() == sat:
    m = s.model()
    for v in m:
        print(v, "=", m[v])
else:
    print("Queries are equivalent.")

SAT? -> sat
Students_name_is_null = False
Students_id_is_null = False
Students_id = 3
Students_present = True
Students_name = "A"
Takes_sid = 4
Takes_present = True
J2_Students_id = 5
J2_Students_id_is_null = True
Takes_sid_is_null = False
J1_Students_id_is_null = False
q1_left_null = False
q1_after_match = False
J2_Takes_sid_is_null = False
q1_after_left_null = False
J1_Takes_sid_is_null = True
q2_after_left_null = False
q2_match = True
q2_after_right_null = False
q2_after_match = True
q2_left_null = False
Takes_cid_is_null = False
J2_Takes_cid_is_null = False
q2_right_null = False
J1_Takes_cid = 0
J1_Students_name_is_null = False
J1_Takes_sid = 0
q1_after_right_null = True
J2_Takes_sid = 4
J1_Takes_cid_is_null = True
q1_match = False
q1_right_null = True
J1_Students_id = 3
J1_Students_name = "A"
Takes_cid = 0
J2_Takes_cid = 0
